# ARIMAX

## Подготовка

In [ ]:
# Работа с данными
import pandas as pd
import numpy as np

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px

# Тесты на стационарность
from statsmodels.tsa.stattools import adfuller, kpss

# Тест на автокорреляцию
from statsmodels.stats.diagnostic import acorr_ljungbox

# Анализ автокорреляции
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Модели ARIMA
from statsmodels.tsa.arima.model import ARIMA

# Декомпозиция
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL, MSTL

In [ ]:
# !pip -q install pmdarima gdown

import gdown
from pmdarima import auto_arima

In [ ]:
def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)

In [ ]:
def kpss_test(timeseries):
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3], index=["Test Statistic", "p-value", "Lags Used"]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value
    print(kpss_output)

## Данные о продажах Favorita

Будем прогнозировать ежедневные продажи сети Favorita в Эквадоре. Представьте, что вы планируете поставки: перед праздником товары нужно привезти заранее, а после него — не остаться с лишними запасами.

В датасете есть дата, город, регион, тип магазина, товарная категория и продажи за 2013–2017 годы. Для занятия возьмём **суммарные продажи сети в тысячах единиц**.

[Описание данных с kaggle](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data) · [Предобработанный CSV с продажами](https://drive.google.com/file/d/1xF5HXH0_fJlSr82A0nRehu6NVeq6mvhR/view)

In [ ]:
gdown.download(id='1xF5HXH0_fJlSr82A0nRehu6NVeq6mvhR', output='train_data.csv', quiet=True)
sales_raw = pd.read_csv('train_data.csv', parse_dates=['date'])

In [ ]:
sales_raw.head()

Предобработаем данные: суммируем продажи по датам и разделим на 1000.

In [ ]:
sales_history = sales_raw.groupby('date')['sales'].sum().sort_index().to_frame('y') / 1000
sales_history.index.name = 'ds'

# Заполняем нулём только пропуски 25 декабря - в Рождество магазин не работал
christmas = (sales_history.index.month == 12) & (sales_history.index.day == 25)
sales_history.loc[christmas, 'y'] = sales_history.loc[christmas, 'y'].fillna(0)

# Др. дыры заполняем через ffill
sales_history = sales_history.asfreq('D').ffill()

In [ ]:
sales = sales_history.loc['2015-01-01':].copy()
sales.head()

Посмотрим, как менялись продажи. Обратите внимание на повторяющиеся колебания и отдельные пики.

In [ ]:
fig = px.line(sales, y='y', title='Ежедневные продажи Favorita')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

Разделим данные по дате: **2015-2016 годы — train, 1 января-15 августа 2017 года — test**. Историю за 2013–2014 годы сохраним для декомпозиции.

Прогнозируем весь test из одной точки — конца 2016 года. Для оперативных закупок горизонт обычно короче, например неделя, а прогноз регулярно обновляется. Здесь длинный горизонт позволяет сравнить поведение моделей.

In [ ]:
split_date = pd.Timestamp('2017-01-01')
train = sales.loc[sales.index < split_date].copy()
test = sales.loc[sales.index >= split_date].copy()

In [ ]:
fig = px.line(title='Обучающая и тестовая выборки')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

На занятиях 5–6 мы начали говорить о регрессорах (предикторах), которые позволяют корректировать прогноз с учётом внешних факторов. Давайте вспомним теорию, а затем перейдём к моделям, которые используют регрессоры.

## Предикторы (регрессоры)

### Dummy-регрессоры

🐎 **Что такое dummy регрессоры?**

Возьмём три признака:

* **Праздник** — дата национального праздника/выходного по поводу праздника.
* **Предпраздничный период** — 3 дня перед ним: покупатели могут закупаться заранее.
* **День выплаты зарплаты** — 15-е число и последний день месяца. Такой график выплат в государственном секторе указан в описании Favorita.

Загрузим календарь с праздниками в Эквадоре и возьмём события уровня `National`.

Учитываем исходные даты праздников (`Holiday`) и даты переноса (`Transfer`). События `Local` и `Additional` пока исключим; 3 предпраздничных дня разметим отдельно.

In [ ]:
gdown.download(id='1cWdWO8CevKUPU-vqU58MhGun6_Z8_9kA', output='holidays_events.csv', quiet=True)
holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
holidays.head()

In [ ]:
holiday_dates = pd.DatetimeIndex(holidays.loc[
    (holidays['locale'] == 'National') & holidays['type'].isin(['Holiday', 'Transfer']), 'date'
].drop_duplicates())

regressors = pd.DataFrame(index=sales.index)
regressors['holiday'] = regressors.index.isin(holiday_dates).astype(int)
regressors.head()

Для прогноза нужен календарь на весь будущий период. Предполагаем, что эти даты известны заранее; неожиданные (непрогнозируемые) события (например, землетрясение), в будущие признаки не включаем.

In [ ]:
fig = px.line(title='Национальные праздники и перенесённые выходные')
fig.add_scatter(x=train.index, y=regressors.loc[train.index, 'holiday'], mode='lines', name='train')
fig.add_scatter(x=test.index, y=regressors.loc[test.index, 'holiday'], mode='lines', name='test')
fig.update_layout(template='plotly_white', width=1000, height=350)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Значение регрессора')
fig.show()

Праздник и подготовка к нему — разные ситуации. Например, продукты к Рождеству могут покупать 22–24 декабря, а не 25-го. Для поставок важен этот сдвиг во времени.

Отметим 3 дня перед праздником. Сам праздник исключим из предпраздничного признака. Также отметим дни выплат.

In [ ]:
pre_holiday_dates = pd.DatetimeIndex([
    date - pd.Timedelta(days=lag)
    for date in holiday_dates for lag in range(1, 4)
])
regressors['pre_holiday'] = (regressors.index.isin(pre_holiday_dates) & ~regressors.index.isin(holiday_dates)).astype(int)
regressors['payday'] = ((regressors.index.day == 15) | regressors.index.is_month_end).astype(int)
dummy_cols = ['holiday', 'pre_holiday', 'payday']
regressors.head()

Посмотрим на признаки и продажи в декабре 2016 года.

In [ ]:
fig = px.line(title='Три дня перед праздниками')
fig.add_scatter(x=train.index, y=regressors.loc[train.index, 'pre_holiday'], mode='lines', name='train')
fig.add_scatter(x=test.index, y=regressors.loc[test.index, 'pre_holiday'], mode='lines', name='test')
fig.update_layout(template='plotly_white', width=1000, height=350)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Значение регрессора')
fig.show()

In [ ]:
calendar_example = train.loc['2016-12-15':'2016-12-31'].join(regressors)
fig = px.line(calendar_example, y='y', title='Продажи перед Рождеством: train, декабрь 2016')
for column, label in [('pre_holiday', 'Перед праздником'), ('holiday', 'Праздник')]:
    points = calendar_example.loc[calendar_example[column] == 1]
    fig.add_scatter(x=points.index, y=points['y'], mode='markers', name=label, marker=dict(size=10))
fig.update_layout(template='plotly_white', width=1000, height=400)
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

### Непрерывные регрессоры

**Непрерывный регрессор** показывает величину, а не наличие события. Например, dummy отмечает, идёт ли акция, а размер скидки показывает, насколько снижена цена.

Для примера непрерывного регрессора загрузим цену нефти из сопутствующего файла Favorita.

In [ ]:
gdown.download(id='1D4rnInuBSKsYkaQl5yi9DKpUuKtrwIAQ', output='oil.csv', quiet=True)
oil = pd.read_csv('oil.csv', parse_dates=['date'])
oil.head()

Цена нефти — возможный экономический фактор для Эквадора, но её полезность для продаж ещё нужно проверить.

Котировки есть не на все дни: заполним пропуски через `ffill`. Будущая цена неизвестна, поэтому для прогноза зададим **сценарий: она остаётся на последнем известном уровне**.

In [ ]:
oil_daily = oil.set_index('date')['dcoilwtico'].sort_index().asfreq('D').ffill()
regressors['oil_price'] = oil_daily.reindex(sales.index)

X_train = regressors.loc[train.index].copy()
X_future = regressors.loc[test.index].copy()

# Сценарий: цена нефти остаётся на последнем известном уровне
X_future['oil_price'] = X_train['oil_price'].iloc[-1]

In [ ]:
fig = px.line(title='Цена нефти: история и сценарий для прогноза')
fig.add_scatter(x=train.index, y=X_train['oil_price'], mode='lines', name='Известные котировки')
fig.add_scatter(x=test.index, y=X_future['oil_price'], mode='lines', name='Сценарий: последний уровень')
fig.update_layout(template='plotly_white', width=1000, height=400)
fig.update_yaxes(title_text='Цена нефти, долл. за баррель')
fig.show()

## Модель ARIMAX

**Формула**

ARIMAX (**A**uto**R**egressive **I**ntegrated **M**oving **A**verage e**X**tended) — это **регрессия с ошибкой, которая моделируется с помощью ARIMA**:

$$y_t = \gamma_1 x_{1,t} + \ldots + \gamma_r x_{r,t} + n_t,$$

где $x_{i,t}$ — регрессоры, $\gamma_i$ — их коэффициенты, а $n_t$ — часть ряда, которую не объяснили регрессоры. Для неё задаём модель ARIMA(p,d,q).

После $d$ (дифференцирования $n_t$) получаем $w_t = \Delta^d n_t$ и описываем его моделью ARMA(p,q):

$$w_t = c + \sum_{i=1}^{p}\alpha_i w_{t-i} + \sum_{j=1}^{q}\beta_j\epsilon_{t-j} + \epsilon_t,$$

где $\epsilon_t$ — белый шум, а $c$ — константа, если она включена в модель. При $d=0$ разности не нужны: $w_t=n_t$.

То есть регрессоры объясняют связь с внешними факторами, а ARIMA — временную зависимость в оставшейся части ряда.

$n_t$ может быть автокоррелированным; $\epsilon_t$ - белый шум.

Буква **X** — дополнительные регрессоры. В нашем примере календарь описывает особые дни, а цена нефти — возможную связь с экономической ситуацией. ARIMA моделирует временные зависимости, оставшиеся после учёта этих факторов.

Для скорости ограничим поиск: $p,q\leq2$, $d\leq1$.

In [ ]:
arima_options

In [ ]:
forecast_size = len(test)
forecast_size

### ARIMAX с dummy-регрессорами

🐎 **Какие порядки (p,d,q) подобраны для ARIMA без регрессоров?**

In [ ]:
arima_model = auto_arima(train['y'], **arima_options)
print(arima_model.summary())
print('Сходимость:', arima_model.arima_res_.mle_retvals['converged'])

forecast_auto_arima = arima_model.arima_res_.forecast(steps=forecast_size, index=test.index).to_frame('y_hat')
forecast_auto_arima

🐎 **Какие порядки (p,d,q) подобраны для ARIMAX (с регрессорами)?**

In [ ]:
dummy_model = auto_arima(train['y'], X=X_train[dummy_cols], **arima_options)
print(dummy_model.summary())
print('Сходимость:', dummy_model.arima_res_.mle_retvals['converged'])

forecast_reg_1 = dummy_model.arima_res_.forecast(steps=forecast_size, index=test.index, exog=X_future[dummy_cols]).to_frame('y_hat')
forecast_reg_1

Сравним прогнозы без регрессоров и с календарными признаками.

In [ ]:
fig = px.line(title='Прогноз продаж Favorita')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_auto_arima.index, y=forecast_auto_arima['y_hat'], mode='lines', name='ARIMA')
fig.add_scatter(x=forecast_reg_1.index, y=forecast_reg_1['y_hat'], mode='lines', name='ARIMAX: календарь')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

🐎 **Что можете сказать о влиянии dummy регрессоров на прогноз?**

С календарными признаками прогноз может различаться для праздника и обычного дня. Но сами колебания ещё не означают улучшения.

Сначала проверим, какие зависимости остались в остатках. Метрики всех моделей сравним в конце занятия.

### Анализ остатков

Остатки помогают увидеть, что модель пока не объяснила.

🐎 **Что такое остатки модели? Как они должны вести себя в идеале?**

Посмотрим на наши остатки.

In [ ]:
dummy_model.resid().plot()

In [ ]:
p, d, q = dummy_model.order
acorr_ljungbox(dummy_model.resid(), lags=[14, 28], model_df=p+q, return_df=True)

В таблице Льюнга–Бокса строка 14 проверяет вместе лаги 1–14, строка 28 — лаги 1–28. Малое p-value указывает на оставшуюся автокорреляцию. `model_df=p+q` учитывает оценённые AR- и MA-параметры.

Есть и другой способ анализировать остатки модели. `plot_diagnostics()` строит четыре графика остатков: динамику, распределение, Q–Q-график и ACF.

In [ ]:
fig = dummy_model.plot_diagnostics(figsize=(10, 8))
plt.show()

**Стандартизированные остатки** — левый верхний график.

$$z_t = \frac{e_t}{s_t},$$

где $s_t$ — оценённое моделью стандартное отклонение ошибки прогноза на один шаг. Делим на стандартное отклонение, **не на дисперсию**.

После стандартизации остатки безразмерны: $z_t=3$ означает отклонение на три стандартных отклонения.

*На что смотреть?*

* Колеблются ли остатки вокруг нуля?
* Есть ли повторяющиеся пики или длинные участки одного знака?
* Меняется ли разброс со временем?

Значения за пределами ±3 — повод проверить необычные события или ошибки данных. Например, модель могла не учесть разовую распродажу. Сам по себе такой выброс не доказывает, что модель плохая.

**Гистограмма с оценкой плотности** — правый верхний график.

Гистограмма показывает, какие остатки встречаются чаще. KDE — плавная оценка их плотности. Для сравнения нанесена кривая стандартного нормального распределения.

Смотрим на асимметрию и тяжёлые хвосты — частые большие отклонения. Они особенно важны для **прогнозных интервалов**: интервалы с предположением нормальности могут оказаться неточными.

Ненормальность сама по себе не доказывает переобучение и не запрещает использовать точечный прогноз ARIMA.

**Нормальный Q–Q-график** — левый нижний.

Сравнивает квантили остатков с квантилями нормального распределения: теоретические — по оси X, выборочные — по оси Y.

*Квантиль уровня $p$* — значение, ниже которого находится примерно доля $p$ наблюдений. Например, квантиль 0,5 — медиана, а 0,25 — первый квартиль. На графике сопоставляются квантили одного уровня.

Если распределения близки, точки идут вдоль опорной прямой. Систематические отклонения показывают различия в форме распределений. При тяжёлых хвостах крайние значения остатков по модулю больше, чем ожидается при нормальном распределении.

**Коррелограмма** — правый нижний график.

Это ACF остатков. Заметная связь на лагах 7 и 14 — повод проверить, достаточно ли учтена недельная сезонность. График помогает найти конкретные лаги, а тест Льюнга–Бокса проверяет их совместно.

### ARIMAX с непрерывными регрессорами

Теперь вместо календаря передадим модели цену нефти. Используем подготовленные значения: историю на train и сценарий постоянной цены на test.

In [ ]:
oil_model = auto_arima(train['y'], X=X_train[['oil_price']], **arima_options)
print(oil_model.summary())
print('Сходимость:', oil_model.arima_res_.mle_retvals['converged'])

forecast_reg_cont = oil_model.arima_res_.forecast(steps=forecast_size, index=test.index, exog=X_future[['oil_price']]).to_frame('y_hat')
forecast_reg_cont

In [ ]:
forecast_reg_cont.head()

In [ ]:
fig = px.line(title='Прогноз продаж Favorita')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_auto_arima.index, y=forecast_auto_arima['y_hat'], mode='lines', name='ARIMA')
fig.add_scatter(x=forecast_reg_1.index, y=forecast_reg_1['y_hat'], mode='lines', name='ARIMAX: календарь')
fig.add_scatter(x=forecast_reg_cont.index, y=forecast_reg_cont['y_hat'], mode='lines', name='ARIMAX: нефть, сценарий')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

🐎 **Как непрерывные регрессоры повлияли на прогноз?**

Здесь проверяется не только модель продаж, но и сценарий будущей цены нефти. Если сценарий окажется далёк от реальности, полезный на train регрессор может не помочь прогнозу.

🐎 **Какие непрерывные регрессоры могут быть полезны для прогнозирования продаж магазинов?**

🐎 Задание на 20 минут

**Спрогнозируйте цену нефти, а затем используйте её прогноз для продаж Favorita.**

**1. Постройте ARIMA-прогноз котировок нефти на даты test.** Подберите порядки с помощью Auto-ARIMA. Для обучения используйте только котировки до начала test; согласуйте их календарь с ежедневными продажами.

**2. Постройте Auto-ARIMAX для продаж с dummy-регрессорами праздников, предпраздничных периодов и дней выплаты зарплаты, а также ценой нефти.** На train используйте исторические котировки, а для прогноза продаж на test — прогноз нефти из первого пункта.

Не используйте фактические будущие котировки или продажи при обучении и построении прогноза!

Данные по нефти находятся в [oil.csv](https://drive.google.com/file/d/1D4rnInuBSKsYkaQl5yi9DKpUuKtrwIAQ/view).

In [ ]:
# your code here

## Как выбирать регрессоры?

**Почему не добавить все признаки сразу?**

* Модель может подогнать шум и переобучиться.
* Сильно связанные признаки мешают разделить их эффекты: например, акция и реклама, которые всегда запускаются вместе.
* Каждый признак нужно подготовить и для будущих дат.

Больше регрессоров — не обязательно точнее прогноз.

Начнём с бизнес-гипотезы: почему признак может быть связан с продажами и будет ли он известен заранее? Затем посмотрим на обучающие данные.

Вернёмся к нефти: сравним её динамику с продажами на train.

In [ ]:
oil.head()

Котировки нефти есть не на все календарные дни, а продажи — ежедневные.

🐎 **Как преобразовать регрессор, чтобы использовать его в ARIMAX?**

In [ ]:
# Переносим только прошлые известные котировки
oil_daily = oil.set_index('date')['dcoilwtico'].sort_index().asfreq('D').ffill()
oil_train = oil_daily.reindex(train.index)
oil_train.head()

In [ ]:
fig = px.line(title='Продажи и цена нефти: разные масштабы')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='Продажи, тыс. ед.')
fig.add_scatter(x=train.index, y=oil_train, mode='lines', name='Нефть, долл. за баррель')
fig.update_layout(template='plotly_white', width=1000, height=400)
fig.show()

Масштабы разные. Разнесём продажи и цену нефти на **две оси Y**, чтобы видеть оба ряда.

Близость линий зависит от настройки осей — это не мера силы связи.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='Продажи', yaxis='y')
fig.add_scatter(x=train.index, y=oil_train, mode='lines', name='Нефть', yaxis='y2')
fig.update_layout(title='Продажи и цена нефти на train', template='plotly_white', width=1000, height=450,
                  yaxis=dict(title='Продажи, тыс. ед.'),
                  yaxis2=dict(title='Нефть, долл. за баррель', overlaying='y', side='right'))
fig.show()

🐎 **Считаете ли вы цену нефти полезным непрерывным регрессором для продаж Favorita?**

Совпадения на графике — только гипотеза. Проверим коэффициенты и качество прогноза с признаком и без него.

**Как отбирать регрессоры?**

1. Смотрим на коэффициенты и p-value. Большая по модулю отрицательная статистика тоже может означать значимость. Незначимость — повод проверить модель без признака, а не автоматически удалить его.
2. Выделяем последние месяцы внутри train для временной валидации. Обучаем варианты с признаком и без него на более ранних данных, сравниваем MAE на одинаковых датах. При близком качестве предпочитаем более простую модель.
3. Выбранный вариант переобучаем на всём train и оцениваем на финальном test. Для более надёжного выбора повторяем валидацию на нескольких периодах.

Так же можно выбирать предпраздничное окно: 3 или 7 дней. Ниже мы разбираем готовый набор признаков; отдельный перебор на валидации в коде не выполняется.

**Прогностическая полезность не равна причинному влиянию.** Улучшение прогноза не доказывает, что изменение регрессора вызывает изменение продаж: он может отражать другие факторы. Для причинного вывода нужны отдельное исследование и дополнительные предпосылки, например корректно спланированный эксперимент.

Посмотрим на коэффициенты. Коэффициент `holiday = 71.5` означает, что модель связывает праздничный день с продажами на 71.5 тыс. единиц выше при прочих равных. Это не доказательство причинного эффекта.

Коэффициент нефти измеряется в тысячах единиц продаж на 1 доллар за баррель; коэффициенты AR и MA безразмерны.

In [ ]:
print(combined_model.summary())

🐎 **Изменилось ли ваше мнение о полезности цены нефти после анализа коэффициентов?**

Связь может проявляться с задержкой. Например, реклама сегодня может быть связана с покупками завтра. Тогда кандидатом в регрессоры будет не только текущий бюджет, но и его лаги — с той же проверкой на валидации.

## Итоги по регрессорам

- Dummy-регрессоры описывают особые периоды через 1/0, непрерывные — величину фактора.
- Будущие значения регрессоров берём из известного плана, прогноза или сценария.
- Полезность проверяем на валидации; причинность — отдельный вопрос.

# SARIMAX

Нашей предыдущей модели не хватает сезонности.

Например, если по выходным продажи выше, одинаковый прогноз на каждый день может привести к нехватке товара в выходные и излишкам в будни. Добавим сезонную часть.

## Те же продажи: добавляем сезонность

Данные и train/test оставим прежними. Начнём с простого ориентира: **повторим продажи последней известной недели** на весь горизонт прогноза.

In [ ]:
# Сезонный наивный прогноз повторяет последнюю известную неделю
seasonal_naive = pd.DataFrame(
    {'y_hat': np.resize(train['y'].iloc[-7:].to_numpy(), len(test))}, index=test.index
)

In [ ]:
seasonal_naive.head()

Сравним средние продажи по дням недели на train.

In [ ]:
weekday_sales = train.groupby(train.index.dayofweek)['y'].mean()
weekday_sales.index = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
weekday_sales

На графике будет проще увидеть, какие дни отличаются по продажам.

In [ ]:
fig = px.bar(weekday_sales, y='y', title='Средние продажи по дням недели: train')
fig.update_layout(template='plotly_white', width=800, height=400)
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

Проверим даты: все прогнозы строим из конца 2016 года.

In [ ]:
fig = px.line(title='Прогноз продаж Favorita')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=seasonal_naive.index, y=seasonal_naive['y_hat'], mode='lines', name='Как в последнюю неделю')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

🐎 **Что можно сказать об этом временном ряде продаж?**

🐎 **Что делать с рядами, в которых есть тренд и сезонность?**

## Декомпозиция

MSTL поможет отдельно рассмотреть тренд, недельный и годовой циклы. Возьмём историю за **2013–2016 годы**: больше повторений годового цикла делает его оценку устойчивее.

Test в декомпозицию не включаем, иначе будущие продажи повлияют на обучающий тренд.

Рассмотрим недельную и годовую сезонность.

🐎 **Какие длины сезонных периодов выбрать?**

In [ ]:
# В декомпозицию не попадают продажи тестового периода
mstl = MSTL(sales_history.loc[:train.index[-1], 'y'], periods=(7, 365))
res = mstl.fit()
fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

🐎 **Как понять, успешна ли декомпозиция?**

In [ ]:
acorr_ljungbox(res.resid, return_df=True)

In [ ]:
adf_test(res.resid)

In [ ]:
kpss_test(res.resid)

🐎 **Что можно сказать о декомпозиции, глядя на остатки?**

## Прогноз тренда

Спрогнозируем тренд через Auto-ARIMA. Это оценка общего направления продаж, полезная для планирования объёмов работы. Для ежедневных поставок её недостаточно: недельные и праздничные пики ещё не учтены.

In [ ]:
trend = res.trend.reset_index()
trend.columns = ['ds','y']
trend = trend.set_index('ds')
trend

In [ ]:
# Обучаем модель на тренде за те же даты, что и исходный train
trend_train = trend.loc[train.index].copy()

In [ ]:
fig = px.line(title="Тренд продаж на обучающем периоде")
fig.add_scatter(x=trend_train.index, y=trend_train['y'], mode='lines', name='train', line=dict(color='blue'))

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="Тренд продаж, тыс. ед.")
fig.show()

In [ ]:
forecast_size = len(test)
forecast_size

In [ ]:
trend_model = auto_arima(trend_train['y'], **arima_options)
print(trend_model.summary())
print('Сходимость:', trend_model.arima_res_.mle_retvals['converged'])
trend_forecast = trend_model.arima_res_.forecast(steps=forecast_size, index=test.index)

In [ ]:
trend_forecast = pd.DataFrame(trend_forecast)
trend_forecast.index = test.index
trend_forecast.columns = ['y_hat']

trend_forecast

На графике — обучающий тренд и его прогноз. «Фактический тренд» test не выделяем: он потребовал бы будущих продаж. Это иллюстрация прогноза компоненты, а не готовый прогноз всего ряда.

In [ ]:
fig = px.line(title="Прогноз тренда продаж")
fig.add_scatter(x=trend_train.index, y=trend_train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=trend_forecast.index, y=trend_forecast['y_hat'], mode='lines', name='forecast', line=dict(color='red'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="Тренд продаж, тыс. ед.")
fig.show()

## Прогноз исходных продаж с учётом сезонности

Теперь другой подход: обучим модель **SARIMA** на исходных данных (до декомпозиции).

**SARIMA** (Seasonal Autoregressive Integrated Moving Average) — ARIMA с сезонной частью:

$$SARIMA(p,d,q)\times(P,D,Q)_m.$$

Напомним обычные параметры:

* **p** — число AR-лагов: зависимость от прошлых значений.
* **d** — число обычных разностей для устранения нестационарности, которую можно убрать дифференцированием.
* **q** — число MA-лагов: зависимость от прошлых случайных ошибок.

**P, D, Q** — соответствующие сезонные порядки, **m** — длина сезона. Для ежедневных продаж с недельным циклом $m=7$.

**Сезонная AR-часть (P)** — связь текущего наблюдения со значениями, отстоящими от него на сезонные лаги (например, со значением 12 месяцев назад).

**Сезонное дифференцирование (D)** — вычитание значения, наблюдавшегося в том же периоде прошлого сезона. Это помогает убрать сезонность и облегчает моделирование.

**Сезонная MA-часть (Q)** работает по тому же принципу, что и несезонная MA-часть, но использует ошибки на сезонных лагах.

*Пример.* Месячные продажи с годовой сезонностью (m = 12).

* **m = 12**: месячные данные с годовым циклом (январь–декабрь).
* **P = 1**: используем продажи из того же месяца прошлого года.
* **D = 1**: делаем сезонное дифференцирование, чтобы сделать ряд стационарным: $Y'_t = Y_t - Y_{t-s}$.
* **Q = 1**: корректируем прогноз с учётом ошибок предыдущего сезона.

Разберём сезонную часть на наших продажах:

* **P = 1** — добавляем AR-связь через 7 дней: например, понедельника с предыдущим понедельником.
* **D = 1** — берём сезонную разность $Y'_t = Y_t - Y_{t-7}$. Она нужна, если сезонная нестационарность требует такого преобразования.
* **Q = 1** — добавляем MA-связь с ошибкой 7 дней назад.

Это примеры значений параметров, а не обязательный набор: `auto_arima` может выбрать нули. Посмотрим на ACF/PACF, затем подберём модель.

In [ ]:
# Два графика рядом
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# ACF (автокорреляция)
plot_acf(train['y'].dropna(), ax=ax[0], lags=20)
ax[0].set_title('ACF')

# PACF (частичная автокорреляция)
plot_pacf(train['y'].dropna(), ax=ax[1], lags=20)
ax[1].set_title('PACF')


plt.tight_layout()
plt.show()

In [ ]:
sarima_options = dict(seasonal=True, m=7, start_p=0, start_q=0, start_P=0, start_Q=0,
                      max_p=2, max_q=2, max_P=1, max_Q=1, max_d=1, max_D=1,
                      maxiter=300, stepwise=True, trace=True)
sarima_model = auto_arima(train['y'], **sarima_options)
print(sarima_model.summary())
print('Сходимость:', sarima_model.arima_res_.mle_retvals['converged'])

In [ ]:
forecast = sarima_model.arima_res_.forecast(steps=forecast_size, index=test.index).to_frame('y_hat')

Сравним SARIMA с правилом «как в последнюю неделю». Затем добавим в сезонную модель уже подготовленные регрессоры — получим SARIMAX.

In [ ]:
forecast.head()

In [ ]:
fig = px.line(title='Прогноз продаж Favorita')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=seasonal_naive.index, y=seasonal_naive['y_hat'], mode='lines', name='Сезонный наивный')
fig.add_scatter(x=forecast.index, y=forecast['y_hat'], mode='lines', name='SARIMA')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Продажи, тыс. ед.')
fig.show()

🐎 Задание на 20 минут

**Спрогнозируйте продажи Favorita с помощью регрессоров.**

**1. Для прогнозов используйте регрессоры, которые мы уже обсуждали (dummy и непрерывные).**

**2. Постройте прогноз с SARIMAX и регрессорами (цена нефти + календарные признаки).**

Используйте тот же train/test. Будущую цену нефти задайте любым сценарием, не подставляя фактические котировки test.

In [ ]:
# your code here

## Итоги по SARIMAX

- MSTL помогла рассмотреть компоненты ряда; прогноз тренда разобрали отдельно.
- SARIMA обучили на исходных продажах с недельным периодом.
- В SARIMAX добавили календарь и сценарий цены нефти.

Теперь проверим, какой из прогнозов точнее.

# Метрики качества

До этого мы изучали графики и остатки на train. Теперь оценим **ошибки прогноза на test**.

Сначала разберём метрики для SARIMA без регрессоров, затем сравним все модели на одинаковых датах. MAE и RMSE будут в тысячах единиц продаж.

## Ошибки, зависящие от масштаба

### MAE и RMSE


$$MAE = \operatorname{mean}(|y_t - \hat{y}_t|)$$
$$RMSE = \sqrt{\operatorname{mean}((y_t - \hat{y}_t)^2)}$$

**MAE** — средняя абсолютная ошибка. **RMSE** сильнее реагирует на крупные промахи. Обе метрики измеряются в единицах ряда; в квадратных единицах измеряется MSE, но не RMSE.

*Пример для магазина.* В обычные дни ошибаемся на 1 тыс. единиц, а перед праздником — на 20 тыс. RMSE сильнее выделит этот промах. Сравним такие ситуации в коде ниже.

Если минимизировать ожидаемую абсолютную ошибку, оптимальный точечный прогноз — медиана прогнозного распределения; для квадратичной ошибки — среднее.

In [ ]:
# Фактические значения
y_actual = np.array([10, 15, 20, 25, 30])

# Прогноз: сценарий 1 — небольшие систематические ошибки
y_pred_small_errors = np.array([11, 14, 21, 24, 29])

# Прогноз: сценарий 2 — один большой выброс
y_pred_outlier = np.array([11, 14, 21, 24, 50])  # выброс в последнем значении

# Считаем MAE и RMSE для небольших ошибок
mae_small = np.mean(np.abs(y_actual - y_pred_small_errors))
rmse_small = np.sqrt(np.mean((y_actual - y_pred_small_errors) ** 2))

# Считаем MAE и RMSE для сценария с выбросом
mae_outlier = np.mean(np.abs(y_actual - y_pred_outlier))
rmse_outlier = np.sqrt(np.mean((y_actual - y_pred_outlier) ** 2))

# Выводим результаты
results = pd.DataFrame({
    "Scenario": ["Small Errors", "Outlier"],
    "MAE": [mae_small, mae_outlier],
    "RMSE": [rmse_small, rmse_outlier]
})


plt.figure(figsize=(8, 6))

# Фактический ряд
plt.plot(y_actual, label="Actual Values", color="black", linewidth=2)

# Прогноз со сплошными мелкими ошибками
plt.plot(y_pred_small_errors, label="Predicted (Small Errors)", linestyle="--", color="blue", linewidth=2)

# Прогноз с выбросом
plt.plot(y_pred_outlier, label="Predicted (With Outlier)", linestyle="--", color="red", linewidth=2)


print(results)

Теперь посчитаем MAE и RMSE для SARIMA на test.

In [ ]:
y_true = test['y']
y_pred = forecast.loc[test.index, 'y_hat']
print('MAE (тыс. ед.): ', (y_true - y_pred).abs().mean())
print('RMSE (тыс. ед.): ', np.sqrt(((y_true - y_pred) ** 2).mean()))

## Ошибки, не зависящие от масштаба

### MAPE

$$MAPE = 100\% \cdot \operatorname{mean}\left(\frac{|y_t - \hat{y}_t|}{|y_t|}\right)$$

**MAPE** — средняя абсолютная процентная ошибка. Например, при факте 100 прогнозы 90 и 110 дают одинаковые 10%. В коде умножаем долю на 100.

Ограничения:

* При нулевых продажах деление не определено; около нуля процентная ошибка нестабильна. Для редко продающихся товаров это особенно важно.
* При положительном факте и неотрицательном прогнозе занижение ограничено 100%, а завышение — нет. Если прогноз отрицательный, ошибка при занижении тоже может превысить 100%.

Ниже — примеры этих случаев. Минимальное значение MAPE — 0%.

In [ ]:
# Фактические значения (y_actual)
y_actual = np.array([10, 20, 30])

# Сценарий 1: сильное завышение прогноза
y_pred_over = np.array([10+100, 20+200, 30+300])

# Сценарий 2: умеренное занижение
y_pred_under = np.array([10-5, 20-10, 30-15])

# Сценарий 3: сильное занижение (уход в отрицательные значения)
y_pred_under_zero = np.array([10-20, 20-30, 30-40])

# Считаем MAPE для всех сценариев
mape_over = np.mean(np.abs((y_actual - y_pred_over) / y_actual)) * 100
mape_under = np.mean(np.abs((y_actual - y_pred_under) / y_actual)) * 100
mape_under_zero = np.mean(np.abs((y_actual - y_pred_under_zero) / y_actual)) * 100

# Выводим результаты
results = pd.DataFrame({
    "Scenario": ["Overestimation", "Underestimation", "Underestimation (negative)"],
    "MAPE (%)": [mape_over, mape_under, mape_under_zero]
})
print(results)

# Визуализируем
plt.figure(figsize=(10, 6))

# Фактический ряд
plt.plot(y_actual, label="Actual Values", color="black", linewidth=2)

# Прогноз с завышением
plt.plot(y_pred_over, label="Forecast (Overestimate)", linestyle="--", color="red", linewidth=2)

# Прогноз с занижением
plt.plot(y_pred_under, label="Forecast (Underestimate)", linestyle="--", color="blue", linewidth=2)

# Прогноз с сильным занижением (отрицательные значения)
plt.plot(y_pred_under_zero, label="Forecast (Underestimate, negative)", linestyle="--", color="green", linewidth=2)

Посчитаем MAPE для того же прогноза SARIMA.

In [ ]:
print('MAPE (%): ', np.mean(np.abs((test['y'] - forecast.loc[test.index, 'y_hat']) / test['y'])) * 100)

### WAPE — взвешенная абсолютная процентная ошибка

$$WAPE = \frac{\sum_t |y_t - \hat{y}_t|}{\sum_t |y_t|} \cdot 100\%.$$

Складываем абсолютные ошибки на test и делим на общий объём фактических продаж за те же даты. Чем меньше WAPE, тем лучше; идеальный прогноз даёт 0%.

*Пример для магазина.* За неделю продали 1 000 единиц, а сумма абсолютных ошибок по дням — 100. Тогда **WAPE = 10%**. Это не означает, что мы ошиблись на 10% в каждый день или потеряли 10% выручки. Завышения и занижения не компенсируют друг друга, потому что берём модули ошибок.

В отличие от MAPE, здесь нет деления на продажи каждого отдельного дня. При ненулевых фактах WAPE можно представить как среднее процентных ошибок с весами, пропорциональными продажам: дни с большим объёмом получают больший вес.


**Пример: почему WAPE может быть удобнее MAPE?**

Сравним прогнозы продаж двух товаров за один период. Продажи обоих товаров считаем в штуках.

| Товар | Факт | Прогноз | Абсолютная ошибка | Процентная ошибка |
|---|---:|---:|---:|---:|
| Редко продающийся | 1 | 3 | 2 | 200% |
| Популярный | 100 | 90 | 10 | 10% |

$$MAPE = \frac{200\% + 10\%}{2} = 105\%.$$

Ошибка всего на 2 единицы у первого товара сильно увеличила MAPE: каждый товар имеет одинаковый вес, независимо от объёма продаж.

$$WAPE = \frac{2 + 10}{1 + 100} \cdot 100\% \approx 11{,}9\%.$$

Суммарная абсолютная ошибка составляет около 12% общего объёма продаж. WAPE удобна, когда важен общий объём ошибок, а небольшие продажи отдельных товаров не должны чрезмерно влиять на процентную метрику. По тому же принципу можно сравнить дни с маленькими и большими продажами одного товара.

Но на фоне популярных товаров WAPE может скрывать плохие прогнозы редких. Если они важны для бизнеса, стоит оценивать их отдельно.

* Отдельные нулевые продажи допустимы. Если продажи равны нулю на всём test, WAPE не определена.
* WAPE может превышать 100%.
* При росте продаж WAPE может снижаться даже при прежней абсолютной ошибке. Поэтому сравниваем модели на одинаковых датах и смотрим также на MAE и MASE, а не только на WAPE ([подробнее у Hyndman](https://robjhyndman.com/hyndsight/wape.html)).

Посчитаем WAPE для того же прогноза SARIMA.

In [ ]:
print('WAPE (%): ', (test['y'] - forecast.loc[test.index, 'y_hat']).abs().sum() / test['y'].abs().sum() * 100)

## Другие метрики

### MDAPE

**MDAPE** (Median Absolute Percentage Error) — модификация MAPE, которая использует медиану абсолютных процентных ошибок вместо среднего. Поэтому отдельные очень большие ошибки влияют на неё меньше.

$$\text{Медианная абсолютная процентная ошибка (MDAPE)} = 100\% \cdot \operatorname{median}\left(\frac{|y_t - \hat{y}_t|}{|y_t|}\right)$$

Проблема нулевых фактических значений остаётся: делить на ноль нельзя. Кроме того, небольшая MDAPE может скрывать редкие, но дорогие для бизнеса ошибки.

In [ ]:

# Пример данных
y_actual = np.array([100, 200, 300, 400, 500])  # фактические значения
y_pred = np.array([110, 210, 280, 350, 5000])    # прогнозируемые значения

# Рассчитаем MAPE
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
print("MAPE (%): ", mape)

# Рассчитаем MDAPE
mdape = np.median(np.abs((y_actual - y_pred) / y_actual)) * 100
print("MDAPE (%): ", mdape)

print("Abs diffs: " , np.abs(y_actual-y_pred))

Посчитаем MDAPE для SARIMA.

In [ ]:
print('MDAPE (%): ', np.median(np.abs((test['y'] - forecast.loc[test.index, 'y_hat']) / test['y'])) * 100)

MDAPE описывает типичную процентную ошибку. Но если редкий крупный промах оставит магазин без товара перед праздником, одной MDAPE недостаточно — смотрим и на RMSE, и на ошибки по отдельным датам.

### MASE

**MASE** (Mean Absolute Scaled Error) — средняя абсолютная масштабированная ошибка. Она показывает величину ошибки модели относительно простого ориентира, рассчитанного на **обучающей выборке**.

$$MASE = \frac{\frac{1}{H}\sum_{h=1}^{H}|y_{T+h}-\hat{y}_{T+h|T}|}{\frac{1}{T-m}\sum_{t=m+1}^{T}|y_t-y_{t-m}|},$$

где $T$ — длина train, $H$ — длина test, $m$ — сезонный период.

* В числителе — MAE нашей модели на test.
* В знаменателе — средняя абсолютная ошибка сезонного наивного правила на train: «значение как в прошлом сезоне», то есть $\hat{y}_t=y_{t-m}$.

Для дневных продаж с недельной сезонностью берём **$m=7$**: сравниваем каждый день обучения с тем же днём предыдущей недели. Для несезонного ряда можно взять $m=1$ — обычное наивное правило «как в предыдущий момент».

MASE не измеряется в процентах и не зависит от единиц ряда. Если знаменатель равен нулю, MASE не определена.

Посчитаем MASE для SARIMA, используя обучающий ориентир с лагом 7.

In [ ]:
m = 7  # недельная сезонность
mae_test = (test['y'] - forecast.loc[test.index, 'y_hat']).abs().mean()
mae_naive_train = train['y'].diff(m).abs().mean()
print('MASE: ', mae_test / mae_naive_train)

MASE < 1 означает, что MAE модели на test меньше средней абсолютной ошибки сезонного наивного правила на train. MASE > 1 — больше, а MASE = 1 — равна этому ориентиру.

*Например*, на train правило «продажи как неделю назад» ошибалось в среднем на 100 единиц, а наша модель на test — на 80. Тогда **MASE = 80 / 100 = 0,8**: ошибка модели на 20% меньше обучающего ориентира. Но обратите внимание, что здесь мы говорим про разные периоды.

Для вывода о качестве по MASE нужно построить оба прогноза из одной точки на тот же тестовый период и сравнить их ошибки.

### SMAPE

**SMAPE** (симметричная средняя абсолютная процентная ошибка) нормирует абсолютную ошибку на величину и факта, и прогноза:

$$SMAPE = 100\% \cdot \operatorname{mean}\left(\frac{2|y_t-\hat{y}_t|}{|y_t|+|\hat{y}_t|}\right).$$

Используем вариант с модулями в знаменателе: его значения лежат от 0% до 200%. Если факт и прогноз одновременно равны нулю, договоримся считать вклад этой пары нулевым.

«Симметричная» означает, что перестановка факта и прогноза не меняет результат. Но равные завышения и занижения относительно одного факта штрафуются **не одинаково**: при факте 100 прогноз 110 даёт около 9,5%, а прогноз 90 — около 10,5%.

SMAPE не решает проблему значений около нуля полностью: небольшие изменения таких значений могут сильно менять процентную ошибку. Поэтому не стоит выбирать её только из-за наличия нулей.

Посчитаем SMAPE для SARIMA.

In [ ]:
y_true = test['y']
y_pred = forecast.loc[test.index, 'y_hat']
denominator = y_true.abs() + y_pred.abs()
# Вклад пары «факт = прогноз = 0» принимаем равным нулю
smape = (2 * (y_true - y_pred).abs() / denominator.replace(0, 1)).mean() * 100
print('SMAPE (%): ', smape)

## Итоги по метрикам

| Метрика | Что показывает | Главное ограничение |
|---|---|---|
| **MAE** | Среднюю абсолютную ошибку в единицах продаж | Не различает недопрогноз и перепрогноз |
| **RMSE** | Ошибку с большим весом крупных промахов | Чувствительна к выбросам |
| **MAPE** | Среднюю абсолютную ошибку в процентах | Нельзя использовать при нулевых фактах; нестабильна около нуля |
| **WAPE** | Сумму абсолютных ошибок относительно общего объёма продаж, в процентах | Не определена при нулевом общем объёме; зависит от уровня продаж в тестовом периоде |
| **MDAPE** | Типичную процентную ошибку | Может скрыть редкие крупные промахи; проблема нулей остаётся |
| **MASE** | Ошибку относительно наивного ориентира на train | Не означает победу над наивным прогнозом на test; не определена при нулевом знаменателе |
| **SMAPE** | Процентную ошибку с нормировкой на факт и прогноз | Чувствительна к значениям около нуля; нужна договорённость для пары нулей |

**Как связать метрики с бизнесом?**

Условный пример: магазин ожидает спрос на 100 упаковок и заказывает ровно по прогнозу. Прогноз 80 создаёт риск дефицита 20 упаковок, а прогноз 120 — риск остатка 20 упаковок. Абсолютная ошибка одинакова, но потери могут различаться: упущенные продажи против хранения или списания.

Поэтому MAE и RMSE — оценка точности, а не прямой расчёт убытков. Для решения о закупке нужны также текущий запас, срок поставки и стоимость дефицита и излишков.

Ниже сравним модели по **MAE, RMSE и WAPE**. На одном и том же test WAPE и MAE дают одинаковый порядок моделей: знаменатель WAPE общий для всех прогнозов.

In [ ]:
forecasts = {'Сезонный наивный': seasonal_naive, 'ARIMA': forecast_auto_arima,
             'ARIMAX: календарь': forecast_reg_1, 'ARIMAX: нефть': forecast_reg_cont,
             'ARIMAX: календарь + нефть': forecast_reg_cont_new,
             'SARIMA': forecast, 'SARIMAX': forecast_sarimax}
total_sales = test['y'].abs().sum()
comparison = []
for name, prediction in forecasts.items():
    errors = test['y'] - prediction.loc[test.index, 'y_hat']
    comparison.append({'Модель': name, 'MAE (тыс. ед.)': errors.abs().mean(),
                       'RMSE (тыс. ед.)': np.sqrt((errors ** 2).mean()),
                       'WAPE (%)': errors.abs().sum() / total_sales * 100 if total_sales > 0 else np.nan})
pd.DataFrame(comparison).set_index('Модель').sort_values('MAE (тыс. ед.)')

🐎 Задание на 10 минут

**Оцените свой прогноз SARIMAX по всем рассмотренным метрикам.**

In [ ]:
# your code here

# ИТОГ

Сегодня прошли весь путь: признаки → ARIMAX → диагностика → сезонность и SARIMAX → оценка прогноза.

- Обсудили dummy-регрессоры (1 — событие, 0 — в остальных случаях) и непрерывные регрессоры (например, чистая прибыль, цена на нефть).
- Применили модели ARIMAX для прогнозирования с регрессорами.
- Разобрали подходы к выбору релевантных регрессоров.
- Рассмотрели модель SARIMAX для сезонных рядов.
- Обсудили метрики, зависящие и не зависящие от масштаба, а также другие метрики качества.
- Поговорили о выборе подходящей метрики качества.

Пожалуйста, оставьте отзыв [здесь](https://forms.gle/yQwoeDHtmn3RM7qDA), чтобы сделать занятия ещё эффективнее :)